# Examining the Relationship between Store Location and Revenue

**Goal:** understand how location characteristics relate to store revenue, and estimate revenue for a prospective site, with honest measures of how reliable the estimates are.

**What this notebook adds over a basic regression:**
1. Data-quality checks
2. Error metrics (RMSE, MAE, R²) compared with a naive baseline
3. 10-fold cross-validation
4. Standardised coefficients, so features can be compared
5. Statistical inference (standard errors, p-values, 95% confidence intervals)
6. Multicollinearity check (VIF)
7. Residual diagnostics
8. A comparison with tree-based models
9. Predictions with 95% intervals and a check that a site lies inside the range of the data

## 1. Imports and settings

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

%matplotlib inline
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

RANDOM_STATE = 100  # same seed as the original analysis, so results are comparable

## 2. Load the data and check its quality

In [ ]:
df = pd.read_csv("location_rev.csv")   # adjust the path if the file lives elsewhere
print("Shape:", df.shape)
display(df.head())

In [ ]:
print("Missing values per column:")
print(df.isna().sum().to_string())
print("\nDuplicate rows:", df.duplicated().sum())

print("\nSummary statistics:")
display(df.describe().T)

# Zeros can be genuine (a brand-new store has age 0) or placeholders for missing data.
# Inspect them and decide.
print("\nRows containing a zero in any column:")
display(df[(df == 0).any(axis=1)])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df["revenue"], bins=30, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of revenue")
axes[0].set_xlabel("Revenue")

sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1])
axes[1].set_title("Correlation matrix")

plt.tight_layout()
plt.show()

## 3. Split the data

Same split as the original analysis (scikit-learn's default 75% train, 25% test, `random_state=100`).
The test set is held back and used only for final evaluation.

In [ ]:
X = df.drop(columns="revenue")
Y = df["revenue"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, random_state=RANDOM_STATE)
print(f"Training rows: {len(X_train)}   Test rows: {len(X_test)}")

## 4. Baseline linear regression

In [ ]:
model = LinearRegression().fit(X_train, Y_train)

coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": model.coef_})
display(coef_df)
print(f"Intercept: {model.intercept_:,.2f}")

## 5. Evaluation with error metrics and a naive baseline

Correlation alone shows how closely predictions *track* the truth, not how *large* the errors are.
RMSE and MAE are in revenue units. R² is the share of revenue variation the model explains.
The naive baseline always predicts the average training revenue, so a useful model must beat it.

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred),
        "Correlation": np.corrcoef(y_true, y_pred)[0, 1],
    }

naive_pred = np.full(len(Y_test), Y_train.mean())
pred_test = model.predict(X_test)

results = pd.DataFrame({
    "Naive (predict the mean)": evaluate(Y_test, naive_pred),
    "Linear regression": evaluate(Y_test, pred_test),
}).T
display(results)

## 6. Cross-validation

A single split can be lucky or unlucky. 10-fold cross-validation fits the model ten times on different
parts of the data and reports the average and spread of the scores.

In [ ]:
cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "R2": "r2",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
}
cv_scores = cross_validate(LinearRegression(), X, Y, cv=cv, scoring=scoring)

cv_summary = pd.DataFrame({
    "Mean": [cv_scores["test_R2"].mean(), -cv_scores["test_RMSE"].mean(), -cv_scores["test_MAE"].mean()],
    "Std. dev. across folds": [cv_scores["test_R2"].std(), cv_scores["test_RMSE"].std(), cv_scores["test_MAE"].std()],
}, index=["R2", "RMSE", "MAE"])
display(cv_summary)

## 7. Standardised coefficients: which feature matters most?

Raw coefficients cannot be compared because the features use different units
(income is in tens of thousands, competitors in single digits).
Standardising puts every feature on the same scale. Each coefficient becomes the change in revenue
for a **one standard deviation** increase in that feature.

In [ ]:
scaler = StandardScaler().fit(X_train)
std_model = LinearRegression().fit(scaler.transform(X_train), Y_train)

# Standardising changes the coefficients' scale but not the predictions
assert np.allclose(std_model.predict(scaler.transform(X_test)), model.predict(X_test))

std_df = (pd.DataFrame({"Feature": X.columns,
                        "Revenue change per +1 SD": std_model.coef_})
          .sort_values("Revenue change per +1 SD", key=np.abs, ascending=False)
          .reset_index(drop=True))
display(std_df)

plt.figure(figsize=(7, 3.5))
sns.barplot(data=std_df, y="Feature", x="Revenue change per +1 SD", color="steelblue")
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Standardised effect on revenue")
plt.tight_layout()
plt.show()

## 8. Statistical inference: standard errors, p-values and confidence intervals

scikit-learn does not report these, so they are computed directly from ordinary least squares theory.
The result matches what `statsmodels` would give.

In [ ]:
def ols_inference(X_df, y, alpha=0.05):
    """OLS with standard errors, t-tests and confidence intervals."""
    Xd = np.column_stack([np.ones(len(X_df)), X_df.to_numpy()])
    names = ["Intercept"] + list(X_df.columns)
    n, k = Xd.shape
    xtx_inv = np.linalg.inv(Xd.T @ Xd)
    beta = xtx_inv @ Xd.T @ y.to_numpy()
    resid = y.to_numpy() - Xd @ beta
    dof = n - k
    sigma2 = resid @ resid / dof                      # residual variance
    se = np.sqrt(np.diag(sigma2 * xtx_inv))
    t_vals = beta / se
    p_vals = 2 * stats.t.sf(np.abs(t_vals), dof)
    t_crit = stats.t.ppf(1 - alpha / 2, dof)

    table = pd.DataFrame({
        "Coefficient": beta,
        "Std. error": se,
        "t": t_vals,
        "p-value": p_vals,
        "CI lower (95%)": beta - t_crit * se,
        "CI upper (95%)": beta + t_crit * se,
    }, index=names)

    ss_res = resid @ resid
    ss_tot = ((y - y.mean()) ** 2).sum()
    r2 = 1 - ss_res / ss_tot
    adj_r2 = 1 - (1 - r2) * (n - 1) / dof
    return table, {"beta": beta, "xtx_inv": xtx_inv, "sigma2": sigma2, "dof": dof,
                   "r2": r2, "adj_r2": adj_r2, "n": n}

ols_table, ols = ols_inference(X_train, Y_train)

# Sanity check: must match scikit-learn
assert np.allclose(ols["beta"], np.r_[model.intercept_, model.coef_])

display(ols_table)
print(f"Training R2 = {ols['r2']:.3f}   Adjusted R2 = {ols['adj_r2']:.3f}   "
      f"Residual std. error = {np.sqrt(ols['sigma2']):,.0f}   (n = {ols['n']}, df = {ols['dof']})")

## 9. Multicollinearity check (variance inflation factor)

If predictors are strongly correlated with each other, coefficients become unstable.
A VIF above about 5 to 10 is a warning sign.

In [ ]:
def vif_table(X_df):
    rows = []
    for col in X_df.columns:
        others = X_df.drop(columns=col)
        r2 = LinearRegression().fit(others, X_df[col]).score(others, X_df[col])
        rows.append({"Feature": col, "VIF": 1 / (1 - r2)})
    return pd.DataFrame(rows)

display(vif_table(X_train))

## 10. Residual diagnostics

A good linear model has residuals that scatter randomly around zero with no pattern,
roughly constant spread, and an approximately normal shape.

In [ ]:
residuals = Y_test - pred_test

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Predicted vs true
axes[0, 0].scatter(pred_test, Y_test, color="grey", alpha=0.7)
lims = [min(pred_test.min(), Y_test.min()), max(pred_test.max(), Y_test.max())]
axes[0, 0].plot(lims, lims, "k-")
axes[0, 0].set_xlabel("Model predictions")
axes[0, 0].set_ylabel("True revenue")
axes[0, 0].set_title("Predicted vs. true revenue (test set)")

# Residuals vs predicted
axes[0, 1].scatter(pred_test, residuals, color="steelblue", alpha=0.7)
axes[0, 1].axhline(0, color="black", linewidth=1)
axes[0, 1].set_xlabel("Model predictions")
axes[0, 1].set_ylabel("Residual (true - predicted)")
axes[0, 1].set_title("Residuals vs. predictions")

# Residual distribution
sns.histplot(residuals, bins=25, kde=True, ax=axes[1, 0], color="steelblue")
axes[1, 0].set_xlabel("Residual")
axes[1, 0].set_title("Distribution of residuals")

# Q-Q plot
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title("Normal Q-Q plot of residuals")

plt.tight_layout()
plt.show()

In [ ]:
# Is the straight-line assumption reasonable for each feature?
# Average residual within 10 bands of each feature. A flat line near zero is what we want.
fig, axes = plt.subplots(1, len(X.columns), figsize=(4 * len(X.columns), 3.5), sharey=True)

for ax, col in zip(axes, X.columns):
    bands = pd.qcut(X_test[col], q=8, duplicates="drop")
    band_means = pd.DataFrame({"x": X_test[col], "res": residuals}).groupby(bands)[["x", "res"]].mean()
    ax.scatter(X_test[col], residuals, alpha=0.25, color="grey")
    ax.plot(band_means["x"], band_means["res"], "o-", color="crimson")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(col, fontsize=10)
axes[0].set_ylabel("Residual")
plt.tight_layout()
plt.show()

## 11. Does a more flexible model do better?

If tree-based models clearly beat linear regression, there are nonlinear effects the linear model misses.
If they do not, the simpler, more interpretable model is the better choice.

In [ ]:
models = {
    "Linear regression": LinearRegression(),
    "Random forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
}

rows = {}
for name, m in models.items():
    m.fit(X_train, Y_train)
    test_scores = evaluate(Y_test, m.predict(X_test))
    cv_r2 = cross_validate(m, X, Y, cv=cv, scoring="r2")["test_score"]
    rows[name] = {"Test RMSE": test_scores["RMSE"], "Test MAE": test_scores["MAE"],
                  "Test R2": test_scores["R2"], "CV R2 (mean)": cv_r2.mean(), "CV R2 (std)": cv_r2.std()}

comparison = pd.DataFrame(rows).T
display(comparison)

## 12. Predictions for a new site, with uncertainty

A single number can mislead. A **95% prediction interval** gives the range in which a single new store's
revenue is expected to fall. The function also warns if the site lies outside the range of the training data,
where the model is extrapolating and should be trusted less.

In [ ]:
def predict_site(site, level=0.95):
    x0 = np.array([1.0] + [site[c] for c in X.columns])
    point = x0 @ ols["beta"]
    leverage = x0 @ ols["xtx_inv"] @ x0
    t_crit = stats.t.ppf(1 - (1 - level) / 2, ols["dof"])
    se_mean = np.sqrt(ols["sigma2"] * leverage)            # uncertainty in the average revenue of such sites
    se_pred = np.sqrt(ols["sigma2"] * (1 + leverage))      # uncertainty for one individual store

    print(f"Predicted revenue: {point:,.0f}")
    print(f"  {level:.0%} confidence interval for the average of such sites: "
          f"{point - t_crit * se_mean:,.0f} to {point + t_crit * se_mean:,.0f}")
    print(f"  {level:.0%} prediction interval for a single new store:       "
          f"{point - t_crit * se_pred:,.0f} to {point + t_crit * se_pred:,.0f}")

    for c in X.columns:
        lo, hi = X_train[c].min(), X_train[c].max()
        pct = stats.percentileofscore(X_train[c], site[c])
        flag = "  <-- OUTSIDE the training range" if not (lo <= site[c] <= hi) else ""
        print(f"  {c:22s} {site[c]:>10,.0f}  (training range {lo:,.0f} to {hi:,.0f}; percentile {pct:.0f}){flag}")
    return point

example_site = {
    "num_competitors": 3,
    "median_income": 30000,
    "num_loyalty_members": 1200,
    "population_density": 2000,
    "location_age": 10,
}
_ = predict_site(example_site)

## 13. Summary of findings

In [ ]:
print("Key results")
print("-" * 60)
print(f"Test-set R2:           {results.loc['Linear regression', 'R2']:.3f}")
print(f"Test-set RMSE:         {results.loc['Linear regression', 'RMSE']:,.0f}   (revenue std. dev. = {Y.std():,.0f})")
print(f"Test-set MAE:          {results.loc['Linear regression', 'MAE']:,.0f}")
print(f"10-fold CV R2:         {cv_summary.loc['R2', 'Mean']:.3f} +/- {cv_summary.loc['R2', 'Std. dev. across folds']:.3f}")
print(f"Largest standardised effect: {std_df.iloc[0]['Feature']}")
print("Features significant at 5%:", ", ".join(ols_table.drop(index='Intercept').query('`p-value` < 0.05').index))
print(f"Best model by test R2: {comparison['Test R2'].idxmax()}")

## 14. What the results show

- **The model explains about 82% of the variation in revenue on unseen data** (test R² 0.821; 10-fold cross-validated R² 0.807 ± 0.030). Typical error is about 3,930 revenue units (MAE), and RMSE is 4,815 against 11,387 for simply predicting the average.
- **All five features are statistically significant** (p < 0.001), and none overlaps with another (VIF about 1), so the coefficients are stable.
- **Median income and location age have the largest effects**, about 6,700 and 6,400 revenue units per standard deviation. Population density follows (about 4,200), then competitors (about -2,200) and loyalty members (about 1,700).
- **The straight-line assumption holds.** Residuals scatter randomly around zero, the average residual stays flat across every feature, and the Q-Q plot is close to a straight line (with a slight right skew).
- **More flexible models did not help.** Random forest (R² 0.745) and gradient boosting (0.762) both did worse than linear regression (0.821), so there is no sign of nonlinear effects being missed.
- **Example site.** Predicted revenue is about 27,600. The average for sites like it is likely between 26,800 and 28,300, but a single new store could land anywhere from roughly 17,500 to 37,700. The site is inside the range of the data, though its population density is low (9th percentile).

**Caveats**
- These are associations, not proven causes. Location age, for instance, may stand in for how established a store is.
- Four rows contain a zero (zero loyalty members, zero population density, zero age, zero competitors) and one revenue sits exactly at 5,000. They may be genuine or placeholders, so check the source.
- The five features are almost completely uncorrelated with each other, which is unusual for real store data and suggests the dataset may be simulated. Confirm its origin before drawing business conclusions.
- The currency or unit of revenue is not stated.